# Hyperparameter Sweep Analysis


Heatmaps showing the delta balanced accuracy (CFT - zero shot) for different hyperparameter combinations:
- Warmup trials
- Window size  
- Finetune epochs

One heatmap per dataset paradigm (MI: Yang2025, P300: BI2015a, SSVEP: Lee2019_SSVEP).

In [ ]:
# imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from glob import glob
import re

GLOBAL_FONTSIZE = 7
LEGEND_FONTSIZE = 6.5

rc_params = {
    "font.size": GLOBAL_FONTSIZE,
    "legend.fontsize": LEGEND_FONTSIZE,
    "axes.titlesize": GLOBAL_FONTSIZE,
    "axes.labelsize": GLOBAL_FONTSIZE,
    "xtick.labelsize": GLOBAL_FONTSIZE,
    "ytick.labelsize": GLOBAL_FONTSIZE,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

DATASETS = {
    "MI": "Yang2025",
    "P300": "BI2015a",
    "SSVEP": "Lee2019_SSVEP",
}

# Aggregated by generate_csv_revision.py from the raw sweep outputs
DATA_CSV = Path("figure_data/hyperparam_sweep_summary.csv")

In [ ]:
# per-configuration summary (mean over subjects of zero-shot / finetuned balanced accuracy)
results_df = pd.read_csv(DATA_CSV)
print(f"Loaded {len(results_df)} experiment results")
results_df.head(10)


In [ ]:
# results_df already contains delta_accuracy from results_detailed.csv
delta_df = results_df.copy()
print(f"Loaded {len(delta_df)} experiment configurations")
delta_df.head(10)

In [ ]:
# create heatmaps for each paradigm
def plot_heatmaps(
    delta_df: pd.DataFrame, fixed_param: str = "finetune_epochs", fixed_value: int = 3
):
    """
    Plot heatmaps of delta accuracy.

    Args:
        delta_df: DataFrame with delta accuracy results
        fixed_param: which parameter to fix ('warmup_trials', 'window_size', or 'finetune_epochs')
        fixed_value: value to fix the parameter at
    """
    if len(delta_df) == 0:
        print("No data to plot")
        return

    paradigms = list(DATASETS.keys())

    with plt.rc_context(rc_params):
        fig, axes = plt.subplots(1, 3, figsize=(6, 2))

        # determine x and y axes based on fixed parameter
        if fixed_param == "finetune_epochs":
            x_param, y_param = "window_size", "warmup_trials"
            x_label, y_label = "Window size", "Warmup trials"
        elif fixed_param == "window_size":
            x_param, y_param = "finetune_epochs", "warmup_trials"
            x_label, y_label = "Finetune epochs", "Warmup trials"
        else:  # warmup_trials
            x_param, y_param = "window_size", "finetune_epochs"
            x_label, y_label = "Window size", "Finetune epochs"

        # filter to fixed parameter value
        filtered_df = delta_df[delta_df[fixed_param] == fixed_value]

        # find global min/max for consistent colorbar, centered at 0
        vmin = filtered_df["delta_accuracy"].min()
        vmax = filtered_df["delta_accuracy"].max()
        # make symmetric around 0 for diverging colormap
        vabs = max(abs(vmin), abs(vmax))
        vmin, vmax = -vabs, vabs

        # create diverging colormap: red-white-green with white at 0
        cmap = plt.cm.RdYlGn

        for idx, paradigm in enumerate(paradigms):
            ax = axes[idx]
            paradigm_df = filtered_df[filtered_df["paradigm"] == paradigm]

            if len(paradigm_df) == 0:
                ax.text(
                    0.5,
                    0.5,
                    "No data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                ax.set_title(f"{paradigm}\n({DATASETS[paradigm]})")
                continue

            # pivot to create heatmap matrix
            pivot = paradigm_df.pivot_table(
                index=y_param, columns=x_param, values="delta_accuracy", aggfunc="mean"
            )

            # plot heatmap with diverging colormap centered at 0
            im = ax.imshow(
                pivot.values, cmap=cmap, aspect="equal", vmin=vmin, vmax=vmax
            )

            # set ticks
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index)

            # add value annotations
            for i in range(len(pivot.index)):
                for j in range(len(pivot.columns)):
                    val = pivot.iloc[i, j]
                    if not np.isnan(val):
                        # use black text for values near white (near 0)
                        color = "black" if abs(val) < vabs * 0.5 else "white"
                        ax.text(
                            j,
                            i,
                            f"{val:.2%}",
                            ha="center",
                            va="center",
                            fontsize=5,
                            color=color,
                        )

            ax.set_title(f"{paradigm} ({DATASETS[paradigm]})")
            ax.set_xlabel(x_label)
            if idx == 0:
                ax.set_ylabel(y_label)
            if idx > 0:
                ax.set_yticklabels([])

        # add colorbar with proper placement
        cbar = fig.colorbar(
            im,
            ax=axes.ravel().tolist(),
            shrink=0.8,
            label="Δ Balanced Accuracy",
            pad=0.02,
        )
        cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))

        plt.suptitle(
            f"CFT vs Zero Shot (fixed {fixed_param}={fixed_value})", fontsize=8, y=1.07
        )
        plt.tight_layout(rect=[0, 0, 0.9, 1])
        plt.savefig(f"hyperparam_heatmap_{fixed_param}_{fixed_value}.pdf")
        plt.show()


if len(delta_df) > 0:
    # plot with different fixed parameters
    plot_heatmaps(delta_df, fixed_param="finetune_epochs", fixed_value=1)
    plot_heatmaps(delta_df, fixed_param="finetune_epochs", fixed_value=3)
    plot_heatmaps(delta_df, fixed_param="finetune_epochs", fixed_value=6)

In [ ]:
# create 3D visualization: one heatmap per finetune_epochs value
def plot_heatmaps_by_epochs(delta_df: pd.DataFrame):
    """Plot heatmaps of warmup vs window for each finetune_epochs value."""
    if len(delta_df) == 0:
        print("No data to plot")
        return

    paradigms = list(DATASETS.keys())
    epochs_values = sorted(delta_df["finetune_epochs"].unique())

    # Set global color limits: -10% to +10%
    vmin, vmax = -0.10, 0.10

    # full-width figure sizing (18cm width like fig4)
    total_w_cm = 18.0
    n_cols = len(epochs_values)
    n_rows = len(paradigms)
    panel_w_cm = total_w_cm / n_cols
    panel_h_cm = panel_w_cm * 0.99
    total_h_cm = panel_h_cm * n_rows

    with plt.rc_context(rc_params):
        fig = plt.figure(figsize=(total_w_cm / 2.54, total_h_cm / 2.54))
        gs = fig.add_gridspec(
            n_rows,
            n_cols,
            wspace=0.3,
            hspace=0.3,
        )

        axes = np.empty((n_rows, n_cols), dtype=object)
        for i in range(n_rows):
            for j in range(n_cols):
                axes[i, j] = fig.add_subplot(gs[i, j])

        for row_idx, paradigm in enumerate(paradigms):
            paradigm_df = delta_df[delta_df["paradigm"] == paradigm]

            for col_idx, epochs in enumerate(epochs_values):
                ax = axes[row_idx, col_idx]
                subset = paradigm_df[paradigm_df["finetune_epochs"] == epochs]

                if len(subset) == 0:
                    ax.axis("off")
                    continue

                pivot = subset.pivot_table(
                    index="warmup_trials",
                    columns="window_size",
                    values="delta_accuracy",
                    aggfunc="mean",
                )

                ax.imshow(
                    pivot.values, cmap="RdYlGn", aspect="auto", vmin=vmin, vmax=vmax
                )

                ax.set_yticks(range(len(pivot.index)))
                ax.set_yticklabels(pivot.index, fontsize=5)

                ax.set_xticks(range(len(pivot.columns)))
                ax.set_xticklabels(pivot.columns, fontsize=5)
                # add text annotations for each cell
                for i in range(len(pivot.index)):
                    for j in range(len(pivot.columns)):
                        val = pivot.iloc[i, j]
                        if not np.isnan(val):
                            color = (
                                "black"
                                if abs(val) < (vmax - vmin) / 2 * 0.5
                                else "white"
                            )
                            ax.text(
                                j,
                                i,
                                f"{val:.1%}",
                                ha="center",
                                va="center",
                                fontsize=4,
                                color=color,
                            )

                if row_idx == 0:
                    ax.set_title(f"Epochs={epochs}", fontsize=6)
                if col_idx == 0:
                    ax.set_ylabel(f"{paradigm}\n\nwarmup", fontsize=6)
                if row_idx == len(paradigms) - 1:
                    ax.set_xlabel("Window size", fontsize=6)

        # plt.suptitle("CFT Improvement over Zero Shot", fontsize=8, y=0.98)
        plt.tight_layout()
        plt.savefig("hyperparam_heatmap_all.pdf", bbox_inches="tight")
        plt.show()


if len(delta_df) > 0:
    plot_heatmaps_by_epochs(delta_df)

In [ ]:
# summary statistics
if len(delta_df) > 0:
    print("Best hyperparameter configurations per paradigm:")
    for paradigm in DATASETS.keys():
        paradigm_df = delta_df[delta_df["paradigm"] == paradigm]
        if len(paradigm_df) == 0:
            continue
        best = paradigm_df.loc[paradigm_df["delta_accuracy"].idxmax()]
        print(f"\n{paradigm} ({DATASETS[paradigm]}):")
        print(
            f"  Best config: warmup={best['warmup_trials']}, window={best['window_size']}, epochs={best['finetune_epochs']}"
        )
        print(f"  Δ Accuracy: {best['delta_accuracy']:.2%}")
        print(f"  CFT Accuracy: {best['cft_accuracy']:.2%}")
        print(f"  Zero Shot Accuracy: {best['zs_accuracy']:.2%}")